In [33]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# =========================
# Datos
# =========================
ai_tools = dataset_limpo_medianag['AISearchDevHaveWorkedWith'].fillna('').astype(str)
ai_tools_exp = ai_tools.str.split(';').explode().str.strip()

ai_tools_exp = ai_tools_exp[
    (ai_tools_exp != '') & 
    (ai_tools_exp != 'Sem dado') & 
    (ai_tools_exp.notna())
]

top_ai = ai_tools_exp.value_counts().head(5).index.tolist()

# =========================
# Co-ocurrencia
# =========================
co_occurrence = {tool: {other: 0 for other in top_ai if other != tool} for tool in top_ai}

for tools_str in ai_tools[ai_tools.str.len() > 0]:
    tools_list = [t.strip() for t in tools_str.split(';') if t.strip() in top_ai]
    
    for i, tool1 in enumerate(tools_list):
        for tool2 in tools_list[i+1:]:
            if tool1 in co_occurrence and tool2 in co_occurrence[tool1]:
                co_occurrence[tool1][tool2] += 1
            elif tool2 in co_occurrence and tool1 in co_occurrence[tool2]:
                co_occurrence[tool2][tool1] += 1

# =========================
# Layout (circular limpio)
# =========================
angles = np.linspace(0, 2*np.pi, len(top_ai), endpoint=False)
pos = {tool: (np.cos(a), np.sin(a)) for tool, a in zip(top_ai, angles)}

# =========================
# Edges
# =========================
edge_x = []
edge_y = []
weights = []

for tool1 in co_occurrence:
    for tool2, weight in co_occurrence[tool1].items():
        if weight > 20:
            x0, y0 = pos[tool1]
            x1, y1 = pos[tool2]
            
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]
            weights.append(weight)

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode='lines',
    line=dict(width=1, color='#94a3b8'),
    hoverinfo='none'
)

# =========================
# Nodes
# =========================
node_x = []
node_y = []
node_text = []

for tool in top_ai:
    x, y = pos[tool]
    node_x.append(x)
    node_y.append(y)
    node_text.append(tool)

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    text=node_text,
    textposition='top center',
    hovertext=node_text,
    hoverinfo='text',
    marker=dict(
        size=50,
        color='#3b82f6',
        line=dict(width=2, color='white')
    )
)

# =========================
# Figura
# =========================
fig = go.Figure(data=[edge_trace, node_trace])

fig.update_layout(
    template="plotly_white",
    showlegend=False,
    hovermode='closest',
    margin=dict(l=20, r=20, t=50, b=20),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
)

# =========================
# Export + show
# =========================
fig.write_html('./Public/assets/ai_tools_network.html')
fig.show()

print(f"✓ Network: {len(top_ai)} nodes | Strong connections (>20) included")

✓ Network: 5 nodes | Strong connections (>20) included
